# F01 — HC3 features (RoBERTa OpenAI detector), stage S2

Runtime: **GPU (T4)**, about 15–20 minutes. Output: `hc3_roberta.parquet` and
`hc3_roberta.manifest.json`. Download both and add them to `data/features/` in the repo
(or send them to Claude). No text is written to the output, only ids, scores and outcomes.

In [ ]:
!pip -q install "d2vi[text] @ git+https://github.com/DRuanli/dvai" transformers accelerate huggingface_hub
import nltk
nltk.download("cmudict", quiet=True); nltk.download("subjectivity", quiet=True)
SUBJ_DIR = nltk.data.find("corpora/subjectivity")
import torch, d2vi
print("d2vi", d2vi.__version__, "| GPU:", torch.cuda.is_available())

## 1. Load HC3 exactly as in the first submission

In [ ]:
from huggingface_hub import hf_hub_download
from d2vi.data.loaders import load_hc3
fp = hf_hub_download("Hello-SimpleAI/HC3", "all.jsonl", repo_type="dataset")
docs = load_hc3(fp)          # 3 largest sources, 1500 per class, random_state=0
print(docs.groupby(["domain", "source"]).size())

## 2. Outcomes (CPU) and detector scores (GPU)

In [ ]:
from d2vi.features.pipeline import outcome_table
from d2vi.features.outcomes import split_halves
from d2vi.features.neural import RobertaOpenAIDetector
texts = docs["text"].tolist()
feats = outcome_table(texts, SUBJ_DIR)
det = RobertaOpenAIDetector()
feats["score"] = det.predict(texts)
feats["score_h1"] = det.predict([split_halves(t)[0] for t in texts])
feats["flag"] = (feats["score"] > 0.5).astype(int)
feats["flag_h1"] = (feats["score_h1"] > 0.5).astype(int)
op = feats.assign(domain=docs["domain"], source=docs["source"]).groupby(["domain", "source"])["flag"].mean()
print(op.unstack().round(3))   # expect s, f close to Table 1 of the first submission

## 3. Save and download

In [ ]:
from d2vi.features.pipeline import save_feature_table, package_versions
import hashlib
manifest = {"corpus": "HC3 all.jsonl", "corpus_sha256": hashlib.sha256(open(fp, "rb").read()).hexdigest(),
            "detector": RobertaOpenAIDetector.model_id, "threshold": 0.5,
            "versions": package_versions("d2vi", "transformers", "torch", "vaderSentiment", "textstat", "nrclex", "scikit-learn")}
path = save_feature_table(docs, feats, "hc3_roberta.parquet", manifest)
from google.colab import files
files.download(str(path)); files.download(str(path.with_suffix(".manifest.json")))